In [0]:
-- 1. Read, Type, Trim, and Deduplicate (Exact Matches Only)
-- Note: Using a batch LIVE VIEW because exact deduplication (DISTINCT) 
-- on a continuous stream requires watermarks which don't apply to dimension updates.
CREATE TEMPORARY LIVE VIEW stg_customers_typed AS
SELECT DISTINCT
  CAST(TRIM(customer_id) AS INT) AS customer_id,
  CAST(TRIM(customer_acct_num) AS BIGINT) AS customer_acct_num,
  TRIM(first_name) AS first_name,
  TRIM(last_name) AS last_name,
  
  -- Handle date casting and nullification of bad values
  COALESCE(
    try_to_date(TRIM(acct_open_date), 'M/d/yyyy'),
    try_to_date(TRIM(acct_open_date), 'yyyy-M-d'),
    try_to_date(TRIM(acct_open_date), 'd-MMM-yy'),
    CAST(TRIM(acct_open_date) AS DATE)
  ) AS acct_open_date,
  
  COALESCE(
    try_to_date(TRIM(birthdate), 'M/d/yyyy'),
    try_to_date(TRIM(birthdate), 'yyyy-M-d'),
    try_to_date(TRIM(birthdate), 'd-MMM-yy'),
    CAST(TRIM(birthdate) AS DATE)
  ) AS birthdate,
  
  TRIM(customer_address) AS customer_address,
  TRIM(customer_city) AS customer_city,
  TRIM(customer_state_province) AS customer_state_province,
  TRIM(customer_country) AS customer_country,
  CAST(TRIM(customer_postal_code) AS STRING) AS customer_postal_code,
  
  UPPER(TRIM(gender)) AS gender,
  UPPER(TRIM(marital_status)) AS marital_status,
  
  TRIM(education) AS education,
  TRIM(occupation) AS occupation,
  TRIM(homeowner) AS homeowner,
  TRIM(yearly_income) AS yearly_income,
  
  CAST(TRIM(num_children_at_home) AS INT) AS num_children_at_home,
  CAST(TRIM(total_children) AS INT) AS total_children,
  TRIM(member_card) AS member_card,
  
  _ingestion_timestamp
FROM mavenmarket.bronze.customers;

-- 2. Run Generic KPI Checks (Completeness, Validity, Consistency)
CREATE TEMPORARY LIVE VIEW stg_customers_checked AS
SELECT 
  *,
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN customer_id IS NULL THEN 'Completeness: customer_id is null; ' ELSE '' END AS err_cust_id,
  
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN birthdate IS NULL THEN 'Completeness: birthdate is null; ' ELSE '' END AS warn_birthdate,
  CASE WHEN acct_open_date IS NULL THEN 'Completeness: acct_open_date is null; ' ELSE '' END AS warn_acct_open,
  CASE WHEN num_children_at_home > total_children THEN 'Consistency: num_children_at_home > total_children; ' ELSE '' END AS warn_children
FROM stg_customers_typed;

-- 3. Consolidate Flags
CREATE TEMPORARY LIVE VIEW stg_customers_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_cust_id)) AS hard_failures,
  TRIM(CONCAT(warn_birthdate, warn_acct_open, warn_children)) AS dq_flags
FROM stg_customers_checked;

-- 4. Write Target Tables (Split the rows)
-- CLEAN / WARN TABLE
CREATE LIVE TABLE silver_customers
COMMENT "Cleaned customers with warn-level data quality flags. Different versions of keys are retained for Gold SCD2."
AS SELECT 
  * EXCEPT(err_cust_id, warn_birthdate, warn_acct_open, warn_children, hard_failures)
FROM stg_customers_flagged
WHERE hard_failures = '';

-- QUARANTINE TABLE
CREATE LIVE TABLE quarantine_customers
COMMENT "Quarantined customers failing hard checks"
AS SELECT 
  * EXCEPT(err_cust_id, warn_birthdate, warn_acct_open, warn_children, dq_flags),
  hard_failures AS quarantine_reasons
FROM stg_customers_flagged
WHERE hard_failures != '';